In [2]:
!pip install -U \
  langchain-ollama \
  langchain-chroma \
  langchain-text-splitters \
  sentence-transformers \
  requests-toolbelt \
   "numpy<2"


Defaulting to user installation because normal site-packages is not writeable


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [4]:
import pandas as pd

books=pd.read_csv("books_cleaned.csv")


In [9]:

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")


In [18]:
books["tagged_description"].to_csv("target_description.txt", sep="\n", index=False,header=False, encoding="utf-8")

In [11]:
import os

# Check current working directory
cwd = os.getcwd()
print(f"📂 Current Directory: {cwd}")

# Check if the file is in this directory
files = os.listdir(cwd)
if "target_description.txt" in files:
    print("✅ Found it! It's right here.")
else:
    print("❌ Not found in this folder. Here's what I DO see:")
    print(files[:10])

📂 Current Directory: C:\Users\user
✅ Found it! It's right here.


In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("target_description.txt", encoding="utf-8")
raw_documents = loader.load()

In [7]:
from langchain_text_splitters import CharacterTextSplitter

# Use a chunk_size that fits your data (e.g., 500 characters)
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,        # Must be > 0
    chunk_overlap=0,       # Set to 0 if you want clean breaks
    is_separator_regex=False
)

documents = text_splitter.split_documents(raw_documents)
print(f"✅ Created {len(documents)} chunks by splitting at \\n.")


Created a chunk of size 1168, which is longer than the specified 500
Created a chunk of size 1214, which is longer than the specified 500
Created a chunk of size 960, which is longer than the specified 500
Created a chunk of size 843, which is longer than the specified 500
Created a chunk of size 881, which is longer than the specified 500
Created a chunk of size 1088, which is longer than the specified 500
Created a chunk of size 1189, which is longer than the specified 500
Created a chunk of size 513, which is longer than the specified 500
Created a chunk of size 752, which is longer than the specified 500
Created a chunk of size 728, which is longer than the specified 500
Created a chunk of size 721, which is longer than the specified 500
Created a chunk of size 1267, which is longer than the specified 500
Created a chunk of size 681, which is longer than the specified 500
Created a chunk of size 553, which is longer than the specified 500
Created a chunk of size 521, which is longe

✅ Created 4415 chunks by splitting at \n.


In [8]:
documents[1]

Document(metadata={'source': 'target_description.txt'}, page_content="9780002261982 A new 'Christie for Christmas' -- a full-length novel adapted from her acclaimed play by Charles Osborne Following BLACK COFFEE and THE UNEXPECTED GUEST comes the final Agatha Christie play novelisation, bringing her superb storytelling to a new legion of fans. Clarissa, the wife of a Foreign Office diplomat, is given to daydreaming. 'Supposing I were to come down one morning and find a dead body in the library, what should I do?' she muses. Clarissa has her chance to find out when she discovers a body in the drawing-room of her house in Kent. Desperate to dispose of the body before her husband comes home with an important foreign politician, Clarissa persuades her three house guests to become accessories and accomplices. It seems that the murdered man was not unknown to certain members of the house party (but which ones?), and the search begins for the murderer and the motive, while at the same time tr

In [10]:
db_books= Chroma.from_documents(
    documents,
    embedding=embeddings
)

In [11]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='1d40032f-8555-43ae-9f39-77fb78ed0a05', metadata={'source': 'target_description.txt'}, page_content='9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.'),
 Document(id='451b79a3-b3a4-44bb-b716-494c7d37b9d1', metadata={'source': 'target_description.txt'}, page_content="9780786808380 Introduce your babies to birds, cats, dogs, and babies through fine art, illustration, and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies each image, introducing the baby to some basic -- and sometimes playful -- information about the subjects."),
 Document(id='c79efdd5-ca65-433f-b759-a606912ba60c', metadata={'source': '

In [12]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_desc,age_book,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,1,25.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [13]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 50)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [14]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_desc,age_book,title_and_subtitle,tagged_description
59,9780007151240,0007151241,The Family Way,Tony Parsons,Parenthood,http://books.google.com/books/content?id=dJEIx...,It should be the most natural thing in the wor...,2005.0,3.51,400.0,2095.0,1,21.0,The Family Way,9780007151240 It should be the most natural th...
143,9780060546571,0060546573,Three Rotten Eggs,Gregory Maguire,Juvenile Fiction,http://books.google.com/books/content?id=t2pWl...,The students of Miss Earth's class in rural Ve...,2005.0,3.74,240.0,76.0,1,21.0,Three Rotten Eggs,9780060546571 The students of Miss Earth's cla...
146,9780060556501,0060556501,"The Lion, the Witch and the Wardrobe (picture ...",C. S. Lewis,Juvenile Fiction,http://books.google.com/books/content?id=FlSpp...,Narnia: A magical land full of wonder and exci...,2004.0,4.19,48.0,5012.0,1,22.0,"The Lion, the Witch and the Wardrobe (picture ...",9780060556501 Narnia: A magical land full of w...
414,9780064405959,0064405958,Seven Spiders Spinning,Gregory Maguire,Juvenile Fiction,http://books.google.com/books/content?id=VPf_u...,"When seven Siberian snow spiders, frozen durin...",1995.0,3.61,144.0,259.0,1,31.0,Seven Spiders Spinning,9780064405959 When seven Siberian snow spiders...
429,9780064434980,0064434982,The Deer in the Wood,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=V7YDW...,Even the youngest child can enjoy a special ad...,1999.0,4.17,32.0,302.0,1,27.0,The Deer in the Wood,9780064434980 Even the youngest child can enjo...
436,9780064471961,0064471969,Shade's Children (rack),Garth Nix,Juvenile Fiction,http://books.google.com/books/content?id=_jlgl...,The Key to Survival Rests in the Hands of Shad...,1998.0,3.90,345.0,10368.0,1,28.0,Shade's Children (rack),9780064471961 The Key to Survival Rests in the...
442,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5S...,"First published more than three decades ago, t...",1998.0,4.39,112.0,1160.0,1,28.0,The Sense of Wonder,9780067575208 First published more than three ...
549,9780131871656,013187165X,Astronomy,Eric Chaisson;Stephen McMillan,Mathematics,http://books.google.com/books/content?id=1O00A...,This introduction to astronomy features an exc...,2006.0,3.85,499.0,153.0,1,20.0,Astronomy: a beginner's guide to the universe,9780131871656 This introduction to astronomy f...
568,9780140139976,0140139974,Sailor Song,Ken Kesey,Fiction,http://books.google.com/books/content?id=-pPSO...,"In Alaska to film a famous children's book, th...",1993.0,3.57,533.0,1956.0,1,33.0,Sailor Song,9780140139976 In Alaska to film a famous child...
706,9780140503449,0140503447,Christmas in Noisy Village,Astrid Lindgren;Ilon Wikland,Juvenile Fiction,http://books.google.com/books/content?id=blqrN...,The noisy children of three neighboring famili...,1981.0,4.18,32.0,797.0,1,45.0,Christmas in Noisy Village,9780140503449 The noisy children of three neig...
